# Latent Semantic Forest LM — Colab 实验入口
核心逻辑全部位于项目 Python 文件。本 Notebook 负责环境、缓存、实验控制、恢复和展示。
先将项目文件放到 `/content/CCG-LM`（或上传包含项目根目录内容的 ZIP）。在 Colab 选择 GPU runtime。
完整实验需要兼容 Blackwell 的 CUDA 版 PyTorch；GPU 型号以真实 runtime 为准。


In [ ]:
import sys, platform, subprocess, shutil, os, json
from pathlib import Path
assert sys.version_info >= (3, 11), sys.version
print(sys.version, platform.platform())
if shutil.which('nvidia-smi'):
    subprocess.run(['nvidia-smi'], check=True)
PROJECT_ROOT = Path('/content/CCG-LM')
if not (PROJECT_ROOT / 'train.py').exists():
    from google.colab import files
    import zipfile
    uploaded = files.upload()
    archives = [Path(name) for name in uploaded if name.endswith('.zip')]
    assert len(archives) == 1, '请上传一个项目 ZIP'
    PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(archives[0]) as archive:
        for member in archive.infolist():
            assert (PROJECT_ROOT / member.filename).resolve().is_relative_to(PROJECT_ROOT.resolve())
        archive.extractall(PROJECT_ROOT)
    if not (PROJECT_ROOT / 'train.py').exists():
        candidates = list(PROJECT_ROOT.glob('*/train.py'))
        assert len(candidates) == 1, 'ZIP 中应包含单个项目目录或根目录内容'
        PROJECT_ROOT = candidates[0].parent
os.chdir(PROJECT_ROOT)
print('Project:', PROJECT_ROOT)


## 安装依赖与真实设备检查
保留满足要求的 Colab PyTorch。若 GPU 运算失败，依据 [PyTorch 官方安装选择器](https://pytorch.org/get-started/locally/) 安装支持本 GPU 的版本并重启 runtime，再继续。无需安装 flash-attn 等额外内核。


In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements.txt'], check=True)
import torch
from utils import device_info
print(json.dumps(device_info(), indent=2))
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
if torch.cuda.is_available():
    dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float32
    q = torch.randn(1, 4, 8, 32, device='cuda', dtype=dtype)
    out = torch.nn.functional.scaled_dot_product_attention(q, q, q, is_causal=True)
    torch.cuda.synchronize()
    assert torch.isfinite(out).all()
    print('CUDA / SDPA check passed:', out.dtype)
else:
    print('CPU 模式仅用于软件 smoke；正式实验需要 GPU runtime。')


## 统一实验控制单元
首先运行默认 smoke。正式训练时设置 `PREPARE_FULL=True, RUN_MAIN=True`，第一轮默认 seed 42、25M 目标 Token。恢复到 100M 时填写 `RESUME_CHECKPOINT` 并设置 `RUN_RESUME=True`；保持模型、Seed、调度与数据配置不变。
`NLL_THRESHOLD` 应在查看实验结果前预先指定。最终测试集只在模型选择结束后使用。


In [ ]:
CONFIG = 'configs/main_125m.yaml'
MODES = ['forest_scalar4', 'forest_vector4']
SEEDS = [42]                       # 后续可用 [13, 42, 2026]
MAIN_BUDGET = 25_000_000
SMOKE_BUDGET = 8192
MICRO_BATCH_SIZE = 4
PREPARE_FULL = False
RUN_MAIN = False
RUN_RESUME = False
RESUME_CHECKPOINT = None           # 完整本地或 Drive 检查点路径
RESUME_BUDGET = 100_000_000
RUN_EVALUATION = True
EVAL_SPLIT = 'dev'                 # 最终模型选择完成后才改为 test
EVAL_BATCH_SIZE = 8
EVAL_MAX_BATCHES = None            # None = 完整 split
DIAGNOSTICS = True
NLL_THRESHOLD = 2.5
MOUNT_DRIVE = False
PERSIST_DATA_CACHE = True          # Drive 缓存仅首次复制，训练读取本地 mmap
OUT_ROOT = Path('/content/lsf_runs')
MAIN_CACHE = PROJECT_ROOT / 'cache/tinystories'
DRIVE_ROOT = Path('/content/drive/MyDrive/latent_semantic_forest')
OUT_ROOT.mkdir(parents=True, exist_ok=True)


## 可选 Drive 挂载
Checkpoint 先原子保存到本地，仅关键节点同步到 Drive。源码、数据缓存与训练状态都不能只存在于 Notebook 变量中。


In [ ]:
if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_ROOT.mkdir(parents=True, exist_ok=True)


## 生成可独立恢复的配置与参数表
修改控制单元后重跑此单元。完整解析后的 YAML 保存在磁盘；训练进程不依赖 Notebook 内存状态。所有模式共享同一缓存与目标顺序。


In [ ]:
import yaml
from utils import load_config
RUNTIME_CONFIGS = PROJECT_ROOT / 'runtime_configs'
RUNTIME_CONFIGS.mkdir(exist_ok=True)
def write_config(mode, seed, smoke=False):
    cfg = load_config('configs/smoke.yaml' if smoke else CONFIG)
    cfg['seed'] = seed
    cfg['model']['mode'] = mode
    cfg['data']['cache_dir'] = str(PROJECT_ROOT / 'cache/demo' if smoke else MAIN_CACHE)
    run_name = f"{cfg['name']}_{mode}_seed{seed}"
    cfg['train']['output_dir'] = str(OUT_ROOT / run_name)
    cfg['train']['budget_tokens'] = SMOKE_BUDGET if smoke else MAIN_BUDGET
    cfg['train']['micro_batch_size'] = MICRO_BATCH_SIZE
    cfg['train']['device'] = DEVICE
    cfg['train']['drive_dir'] = str(DRIVE_ROOT / run_name) if MOUNT_DRIVE else None
    path = RUNTIME_CONFIGS / f'{run_name}.yaml'
    path.write_text(yaml.safe_dump(cfg, allow_unicode=True), encoding='utf-8')
    return path
SMOKE_CONFIG = write_config('forest_vector4', 42, smoke=True)
MAIN_CONFIGS = [write_config(mode, seed) for mode in MODES for seed in SEEDS]
for name in ['smoke', 'main_125m', 'scale_245m']:
    subprocess.run([sys.executable, 'experiments.py', '--config', f'configs/{name}.yaml',
                    '--parameters-only', '--output-dir', str(OUT_ROOT / 'parameters' / name)], check=True)


## 数据下载与预处理
Smoke 使用明确标记的合成数据。正式缓存从 TinyStories 官方 train/validation 构建；首次运行可能耗时。若 Drive 中已有缓存，直接复制到本地，避免重做分句和 BPE。


In [ ]:
subprocess.run([sys.executable, '-m', 'data.prepare', '--config', str(SMOKE_CONFIG), '--demo'], check=True)
if PREPARE_FULL or RUN_MAIN or RUN_RESUME:
    drive_cache = DRIVE_ROOT / 'data_cache'
    if MOUNT_DRIVE and PERSIST_DATA_CACHE and (drive_cache / 'manifest.json').exists() and not (MAIN_CACHE / 'manifest.json').exists():
        shutil.copytree(drive_cache, MAIN_CACHE, dirs_exist_ok=True)
    subprocess.run([sys.executable, '-m', 'data.prepare', '--config', str(MAIN_CONFIGS[0]), '--verify-cache'], check=True)
    if MOUNT_DRIVE and PERSIST_DATA_CACHE and not (drive_cache / 'manifest.json').exists():
        shutil.copytree(MAIN_CACHE, drive_cache, dirs_exist_ok=True, ignore=shutil.ignore_patterns('manifest.json'))
        from utils import sync_file
        sync_file(MAIN_CACHE / 'manifest.json', drive_cache)


## 分阶段单元测试
包含独立 CKY 全枚举、因果性、Mask、梯度、全部模式、故事边界和完整恢复。CUDA 可用时额外执行 BF16/SDPA 测试。


In [ ]:
subprocess.run([sys.executable, '-m', 'pytest', '-q'], check=True)


## 小模型完整 Smoke
若运行目录已经有日志，请使用恢复单元或更换 OUT_ROOT，避免覆盖已有实验。


In [ ]:
subprocess.run([sys.executable, 'train.py', '--config', str(SMOKE_CONFIG)], check=True)


## 正式主实验
默认只支持单 GPU 顺序执行。相同 Seed 在所有模式采用相同目标数据顺序。预算与学习率参考终点独立。


In [ ]:
if RUN_MAIN:
    assert torch.cuda.is_available(), '正式实验请选择 GPU runtime'
    for cfg_path in MAIN_CONFIGS:
        subprocess.run([sys.executable, 'train.py', '--config', str(cfg_path)], check=True)
else:
    print('正式训练未启动；修改控制单元 RUN_MAIN=True 后运行。')


## 从磁盘 Checkpoint 恢复
可以从 Drive 复制文件到本地再填写路径。模型、优化器、RNG、数据游标和累计 Token 都从文件恢复；不会重启 warmup。latest.pt 可能比最近的 tokens_*.pt 旧，应选择已完成的最大 Token 节点。


In [ ]:
if RUN_RESUME:
    assert RESUME_CHECKPOINT is not None and Path(RESUME_CHECKPOINT).exists()
    checkpoint = torch.load(RESUME_CHECKPOINT, map_location='cpu', weights_only=False)
    resume_cfg = checkpoint['config']
    resume_cfg['data']['cache_dir'] = str(MAIN_CACHE)
    run_name = f"{resume_cfg['name']}_{resume_cfg['model']['mode']}_seed{resume_cfg['seed']}"
    resume_cfg['train']['output_dir'] = str(OUT_ROOT / run_name)
    resume_cfg['train']['drive_dir'] = str(DRIVE_ROOT / run_name) if MOUNT_DRIVE else None
    resume_cfg['train']['device'] = DEVICE
    resume_path = RUNTIME_CONFIGS / 'resume.yaml'
    resume_path.write_text(yaml.safe_dump(resume_cfg), encoding='utf-8')
    del checkpoint
    assert (MAIN_CACHE / 'manifest.json').exists(), '请先恢复同一个数据缓存'
    subprocess.run([sys.executable, 'train.py', '--config', str(resume_path), '--resume', str(RESUME_CHECKPOINT),
                    '--budget-tokens', str(RESUME_BUDGET)], check=True)


## 批量评估与探索性语义测试
正常记忆、关闭记忆、跨故事打乱记忆和 Forest Top-1 使用同一检查点。模板以 BPE 长度过滤并反向配对；它们不是人工标准语义标注集。


In [ ]:
if RUN_EVALUATION:
    for run_dir in sorted(OUT_ROOT.iterdir()):
        checkpoints = sorted(run_dir.glob('tokens_*.pt'), key=lambda p: int(p.stem.split('_')[1]))
        if not checkpoints:
            continue
        cfg = json.loads((run_dir / 'config.json').read_text())
        cmd = [sys.executable, 'evaluate.py', '--checkpoints', *map(str, checkpoints),
               '--cache-dir', cfg['data']['cache_dir'], '--output-dir', str(run_dir / 'evaluation'),
               '--split', EVAL_SPLIT, '--batch-size', str(EVAL_BATCH_SIZE), '--device', DEVICE, '--top1']
        if DIAGNOSTICS:
            cmd += ['--diagnostics']
        if EVAL_MAX_BATCHES is not None:
            cmd += ['--max-batches', str(EVAL_MAX_BATCHES)]
        subprocess.run(cmd, check=True)
        subprocess.run([sys.executable, 'semantic_tests.py', '--cache-dir', cfg['data']['cache_dir'],
                        '--checkpoint', str(checkpoints[-1]), '--output-dir', str(run_dir / 'semantic_tests'),
                        '--device', DEVICE], check=True)


## 学习曲线、结果矩阵与 GPU 成本
AULC 仅在覆盖 5M–100M 后报告。CPU 运行不填 GPU 小时/显存。小规模 smoke 的学习曲线仅检查绘图流程。


In [ ]:
import pandas as pd
from IPython.display import display, Image
logs = sorted(OUT_ROOT.glob('*/metrics.csv'))
if logs:
    subprocess.run([sys.executable, 'evaluate.py', '--logs', *map(str, logs), '--output-dir', str(OUT_ROOT / 'curves'),
                    '--nll-threshold', str(NLL_THRESHOLD)], check=True)
    display(Image(filename=str(OUT_ROOT / 'curves/learning_curves.png')))
    display(pd.read_csv(OUT_ROOT / 'curves/learning_efficiency.csv'))
summaries = [pd.read_csv(path) for path in OUT_ROOT.glob('*/evaluation/summary.csv')]
if summaries:
    summary = pd.concat(summaries, ignore_index=True)
    summary.to_csv(OUT_ROOT / 'evaluation_matrix.csv', index=False)
    display(summary)


## 按故事配对 Bootstrap
填入两个同一 split、同一数据缓存的 `*_stories.csv`。在多 Seed 实验中先按 Seed 配对报告；故事 Bootstrap 不替代独立训练 Seed 的不确定性。


In [ ]:
STORIES_A = None
STORIES_B = None
if STORIES_A and STORIES_B:
    subprocess.run([sys.executable, 'evaluate.py', '--bootstrap', STORIES_A, STORIES_B,
                    '--output-dir', str(OUT_ROOT / 'bootstrap')], check=True)
